# 🚀 Модуль 8. Урок 1. Фундамент промпт-инжиниринга: System Prompts, Few-Shot и ролевые контексты

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_08_prompt_engineering/lesson_01_prompt_engineering_fundamentals.ipynb)

## 📌 Бизнес-контекст и методология
Промпт-инжиниринг для аналитика — это проектирование детерминированных текстовых пайплайнов.
Основные компоненты:
1. Ролевой контекст (System Prompt).
2. Четкие правила форматирования (JSON / Таблицы).
3. ПримерыFew-Shot для устранения двусмысленности.

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

# Загрузка ключевых датасетов
customers_csv = ensure_dataset("customers.csv", subfolder="raw")
orders_csv = ensure_dataset("orders.csv", subfolder="raw")
saas_csv = ensure_dataset("saas_users.csv", subfolder="raw")
print("🟢 Окружение готово к анализу данных.")


🟢 Окружение готово к анализу данных.


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
import json

prompt_template = {
    "role": "system",
    "content": "Ты Senior Data Analyst. Классифицируй отзывы клиентов на категории: [Bug, Billing, Feature]. Формат ответа: строго JSON.",
    "examples": [
        {"input": "Списали деньги дважды за подписку!", "output": {"category": "Billing", "severity": "HIGH"}},
        {"input": "Кнопка экспорта в CSV не нажимается", "output": {"category": "Bug", "severity": "MEDIUM"}}
    ]
}

print(json.dumps(prompt_template, ensure_ascii=False, indent=2))


{
  "role": "system",
  "content": "Ты Senior Data Analyst. Классифицируй отзывы клиентов на категории: [Bug, Billing, Feature]. Формат ответа: строго JSON.",
  "examples": [
    {
      "input": "Списали деньги дважды за подписку!",
      "output": {
        "category": "Billing",
        "severity": "HIGH"
      }
    },
    {
      "input": "Кнопка экспорта в CSV не нажимается",
      "output": {
        "category": "Bug",
        "severity": "MEDIUM"
      }
    }
  ]
}


### 🎯 Практическое задание 1: Валидация структуры промпта
Убедитесь, что промпт содержит описание роли и примеры.

In [3]:
### ВАШ КОД ЗДЕСЬ
assert "role" in prompt_template
assert len(prompt_template["examples"]) >= 2
print("✅ Тест успешно пройден: архитектура промпта валидна!")


✅ Тест успешно пройден: архитектура промпта валидна!


### 💡 Zero-Shot vs Few-Shot
Few-Shot примеры сокращают частоту отклонений от требуемой схемы вывода на 85%.